In [1]:
import numpy as np
import os
import matplotlib.pyplot as plt
import implementations as impl

from itertools import combinations_with_replacement

In [2]:
# load preprocessed data
DATA_DIR = os.path.expanduser("dataset")
data_file = os.path.join(DATA_DIR, "preprocessed.npz")

d = np.load(data_file)
x_train, x_test, y_train, train_ids, test_ids, names = (
    d["x_train"], d["x_test"], d["y_train"], d["train_ids"], d["test_ids"], d["names"])

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test: ", x_test.shape)

x_train: (328135, 133)
y_train: (328135,)
x_test:  (109379, 133)


### Least squares

We compute the least squares method on the entire dataset

In [3]:
w_ls, loss_ls = impl.least_squares(y_train, x_train)
print("RMSE loss:", np.sqrt(2 * loss_ls))

RMSE loss: 319162.10827337235


We now study the effect of augmented feature vectors.

For `degree > 1` we use a smaller subset of the data as the length of the feature vectors blow up very fast.

In [4]:
def augment_features(X, d):
    N, p = X.shape
    features = [np.ones(N)]

    for degree in range(1, d + 1):
        for indices in combinations_with_replacement(range(p), degree):
            features.append(np.prod(X[:, indices], axis=1))

    return np.column_stack(features)

In [5]:
def split_data(x, y, ratio, seed=1):
    # set seed
    np.random.seed(seed)
    N = x.shape[0]
    split_idx = int(N * ratio)

    randomize = np.random.permutation(N)
    x, y = x[randomize], y[randomize]

    return x[:split_idx], x[split_idx:], y[:split_idx], y[split_idx:]

In [6]:
def train_splits(x, y, degree, ratio, seed, reduction=10):
    if degree >= 2:
        # choose smaller subset of data
        x = x[:len(x) // reduction]
        y = y[:len(y) // reduction]
    
    x_tr, x_te, y_tr, y_te = split_data(x, y, ratio, seed)
        
    phi_tr = augment_features(x_tr, degree)
    phi_te = augment_features(x_te, degree)

    weights, mse_tr = impl.least_squares(y_tr, phi_tr)

    rmse_tr = np.sqrt(2 * mse_tr)
    rmse_te = np.sqrt(2 * impl.mse_loss(y_te, phi_te, weights))
    
    print(
        "proportion={p}, degree={d}, Training RMSE={tr:.3f}, Testing RMSE={te:.3f}".format(
            p=ratio, d=degree, tr=rmse_tr, te=rmse_te
        )
    )

In [36]:
seed = 6
split_ratios = [0.1, 0.5, 0.7, 0.9]
degrees = [1, 2]

for ind, degree in enumerate(degrees):
    for ind_r, split_ratio in enumerate(split_ratios):
        train_splits(x_train, y_train, degree, split_ratio, seed)
    print()

proportion=0.1, degree=1, Training RMSE=16962.806, Testing RMSE=153271.985
proportion=0.5, degree=1, Training RMSE=85089.233, Testing RMSE=84829.246
proportion=0.7, degree=1, Training RMSE=119013.051, Testing RMSE=50883.768
proportion=0.9, degree=1, Training RMSE=152910.248, Testing RMSE=16970.863

proportion=0.1, degree=2, Training RMSE=0.000, Testing RMSE=5437776.823
proportion=0.5, degree=2, Training RMSE=5610.435, Testing RMSE=940861360.568
proportion=0.7, degree=2, Training RMSE=8866.020, Testing RMSE=1016241426.893
proportion=0.9, degree=2, Training RMSE=11998.321, Testing RMSE=8828615.021



### Ridge regression

In [8]:
def ridge_regression_fit(x, y, degree, ratio, seed, reduction=10):
    if degree >= 2:
        x = x[:len(x) // reduction]
        y = y[:len(y) // reduction]
        
    
    lambdas = np.logspace(-5, 0, 15)

    x_tr, x_te, y_tr, y_te = split_data(x, y, ratio, seed)

    phi_tr = augment_features(x_tr, degree)
    phi_te = augment_features(x_te, degree)

    rmse_tr = []
    rmse_te = []
    for ind, lambda_ in enumerate(lambdas):
        weights = impl.ridge_regression(y_tr, phi_tr, lambda_)

        rmse_tr.append(np.sqrt(2* impl.mse_loss(y_tr, phi_tr, weights) ))
        rmse_te.append(np.sqrt(2* impl.mse_loss(y_te, phi_te, weights) ))
        
        print(
            "proportion={p}, degree={d}, lambda={l:.3f}, Training RMSE={tr:.3f}, Testing RMSE={te:.3f}".format(
                p=ratio, d=degree, l=lambda_, tr=rmse_tr[ind], te=rmse_te[ind]
            )
        )
    #plot_train_test(rmse_tr, rmse_te, lambdas, degree)

In [9]:
seed = 64
split_ratio = 0.7
degrees = [1,2]

for ind, degree in enumerate(degrees):
    ridge_regression_fit(x_train, y_train, degree, split_ratio, seed)
    print()

proportion=0.7, degree=1, lambda=0.000, Training RMSE=118515.740, Testing RMSE=51376.921
proportion=0.7, degree=1, lambda=0.000, Training RMSE=118515.740, Testing RMSE=51376.923
proportion=0.7, degree=1, lambda=0.000, Training RMSE=118515.743, Testing RMSE=51376.927
proportion=0.7, degree=1, lambda=0.000, Training RMSE=118515.755, Testing RMSE=51376.924
proportion=0.7, degree=1, lambda=0.000, Training RMSE=118515.812, Testing RMSE=51376.885
proportion=0.7, degree=1, lambda=0.001, Training RMSE=118516.062, Testing RMSE=51376.722
proportion=0.7, degree=1, lambda=0.001, Training RMSE=118517.198, Testing RMSE=51376.383
proportion=0.7, degree=1, lambda=0.003, Training RMSE=118522.579, Testing RMSE=51376.646
proportion=0.7, degree=1, lambda=0.007, Training RMSE=118548.678, Testing RMSE=51383.268
proportion=0.7, degree=1, lambda=0.016, Training RMSE=118675.311, Testing RMSE=51427.534
proportion=0.7, degree=1, lambda=0.037, Training RMSE=119266.173, Testing RMSE=51658.464
proportion=0.7, degre